# Module 1 Exercise: Partitioning, OPTIMIZE, ZORDER, VACUUM

SQL Server analogues:

| Delta operation | SQL Server equivalent |
|---|---|
| Partitioning | Partition scheme + partition function |
| OPTIMIZE | `ALTER INDEX ... REORGANIZE`, shrink + compact |
| ZORDER BY | Clustered index key selection (data skipping stats) |
| VACUUM | Ghost record cleanup / tempdb reclaim |

DataBuilder wrote `orders` in 22 append batches, partitioned by year/month/day.
That means many small files per partition, which is exactly what OPTIMIZE fixes.
We will measure the effect by watching the file count shrink.

## 1. Baseline: what does the table look like on disk

`DESCRIBE DETAIL` is the Delta equivalent of `sp_spaceused` + `sys.partitions`:
file count, total size, partition columns, location.

In [ ]:
%%sql
DESCRIBE DETAIL orders

In [ ]:
%%sql
SHOW PARTITIONS orders LIMIT 20

## 3. Count files *before* OPTIMIZE

If tables is fragmented, the files count before optimize than after.

In [ ]:
# PySpark is useful here: inputFiles() shows exactly which files Spark will read.
df = spark.table("customers")
files_before = df.inputFiles()
print(f"Files read for full table (before OPTIMIZE): {len(files_before):,}")

## 4. OPTIMIZE - compact small files

Equivalent to reorganising a fragmented clustered index. Files below the target
size (default 1 GB for Delta) are rewritten into fewer, larger files. The
logical table is unchanged, only the physical layout.

In [ ]:
%%sql
OPTIMIZE orders

In [ ]:
%%sql
DESCRIBE DETAIL orders

Compare `numFiles` and `sizeInBytes` with the pre-OPTIMIZE values from step 1.
Fewer, larger files means fewer file opens, better column statistics, faster
scans. The OPTIMIZE result row itself also reports `numFilesAdded` and
`numFilesRemoved`.

### Measure the baseline file count

`OPTIMIZE` compacted small files, but all files in partitions that still match
our filter will be read unless data skipping can eliminate them. Count the files
Spark actually opens for a single-customer lookup: this is the number ZORDER is
going to reduce.


In [ ]:
# PySpark is useful here: inputFiles() shows exactly which files Spark will read.
df = spark.table("orders").filter("customer_id = 1234567")
files_before = df.inputFiles()
print(f"Files read for customer_id = 1234567 (before ZORDER): {len(files_before):,}")


## 5. ZORDER BY - colocate data by a non-partition column

Think of this as choosing the key for a clustered index. Rows with similar
values end up in the same file, so Delta's per-file min/max stats can skip most
of the files when you filter by that column. Great for high-cardinality columns
you filter by often.

In [ ]:
%%sql
OPTIMIZE orders ZORDER BY (customer_id)

### Measure the file count after ZORDER

Same query, same predicate. The row count is identical (no data changed), but
the file count should drop sharply because rows for `customer_id = 1234567` are
now colocated into a small set of files, and Delta's min/max stats can skip the
rest. This is data skipping, the Delta equivalent of an index seek.


In [ ]:
df = spark.table("orders").filter("customer_id = 1234567")
files_after = df.inputFiles()
print(f"Files read for customer_id = 1234567 (after ZORDER):  {len(files_after):,}")
print(f"Reduction: {len(files_before) - len(files_after):,} fewer files")


In [ ]:
%%sql
EXPLAIN FORMATTED
SELECT COUNT(*) FROM orders
WHERE  customer_id = 1234567
AND    year = 2024 AND month = 6

Check the plan: the `PushedFilters` list and the `PartitionFilters` are the
same, but the *number of files read* should be lower than before the ZORDER.
That is data skipping at work, the Delta equivalent of an index seek picking
off just the leaf pages it needs.

## 6. VACUUM - remove old files

OPTIMIZE rewrote your data into new files, but the old files are still on disk
so time travel (`VERSION AS OF`) keeps working. VACUUM is the cleanup pass that
deletes anything older than the retention window (default 7 days).

Always run `DRY RUN` first: it lists what *would* be deleted without touching a
byte. This is the Fabric equivalent of reviewing a shrink plan before executing.

In [ ]:
%%sql
VACUUM orders RETAIN 168 HOURS DRY RUN

## Enforce clean up

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM orders RETAIN 0 HOURS DRY RUN

The result lists every file path scheduled for deletion. Running `VACUUM orders`
without `DRY RUN` and with the default 168-hour (7-day) retention is safe. Going
below 168 hours requires setting
`spark.databricks.delta.retentionDurationCheck.enabled = false` and will break
time travel for any version older than the new window.

## Key takeaways

1. **Partitioning** is a physical layout decision, pick low-cardinality columns
   you filter on (year, month, region). Same rules as SQL Server partitioned
   tables.
2. **OPTIMIZE** compacts small files, same intent as index reorganisation.
3. **ZORDER** picks a clustering key for data skipping, same intent as choosing
   the clustered index key.
4. **VACUUM** is your cleanup, but it also draws a line in the sand for time
   travel, so pick the retention window deliberately.